## Welcome to your notebook.


#### Run this cell to connect to your GIS and get started:

In [4]:
from arcgis.gis import GIS
gis = GIS("home")

You are logged on as carver.rasmussen.gis with an administrator role, proceed with caution.


#### Now you are ready to start!

In [5]:
import urllib.request, zipfile, io, os, shutil
url = "https://github.com/MegaDeadCowboy/whatcom-creek-salmon-monitoring/archive/refs/heads/main.zip"
shutil.rmtree("/arcgis/home/repo", ignore_errors=True)
zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall("/arcgis/home")
os.rename("/arcgis/home/whatcom-creek-salmon-monitoring-main", "/arcgis/home/repo")

In [6]:
%cd /arcgis/home/repo
!python scripts/build_xlsform.py docs/schema.yaml form/whatcom_salmon.xlsx

/arcgis/home/repo
Wrote form/whatcom_salmon.xlsx (15 fields, version 2026100221)


In [7]:
import pandas as pd
pd.read_excel("form/whatcom_salmon.xlsx", sheet_name="choices").query("list_name == 'reach'")

,list_name,name,label
0,reach,R1,R1: Mouth to Maritime Heritage Park
1,reach,R2,R2: Maritime Heritage Park to Woburn St
2,reach,R3,R3: Woburn St to WWU Fisheries Rental
3,reach,R4,R4: WWU Rental to Lake Outlet


In [8]:
from arcgis.gis import GIS
from arcgis.apps.survey123 import SurveyManager
gis = GIS("home")
forms = gis.content.search('title:"Whatcom Creek Salmon Survey"', item_type="Form")
print(forms)

[<Item title:"Whatcom Creek Salmon Survey" type:Form owner:carver.rasmussen.gis>]


You are logged on as carver.rasmussen.gis with an administrator role, proceed with caution.


In [14]:
sm = SurveyManager(gis)
s = sm.get(forms[0].id)
s.publish(xlsform="/arcgis/home/repo/form/whatcom_salmon.xlsx")

<Survey @ Whatcom Creek Salmon Survey>

In [10]:
import yaml
schema = yaml.safe_load(open("/arcgis/home/repo/docs/schema.yaml"))
svc = [i for i in gis.content.search('title:"Whatcom Creek Salmon Survey"')
       if i.type == "Feature Service" and "View Service" not in i.typeKeywords]
print(svc)  # expect exactly 1
lyr = svc[0].layers[0]
updates = []
for f in lyr.properties.fields:
    sf = next((x for x in schema["fields"] if x["name"] == f["name"] and x.get("domain")), None)
    if sf and f.get("domain"):
        d = dict(f["domain"])
        d["codedValues"] = [{"name": v, "code": k} for k, v in schema["domains"][sf["domain"]].items()]
        updates.append({"name": f["name"], "domain": d})
print(lyr.manager.update_definition({"fields": updates}))

[<Item title:"Whatcom Creek Salmon Survey" type:Feature Layer Collection owner:carver.rasmussen.gis>]
{'success': True}


In [11]:
svc[0].homepage

'https://www.arcgis.com/home/item.html?id=1d616f8cf70d4384ade27ee41b53de19'

In [12]:
print([f["name"] for f in svc[0].layers[0].properties.fields])

['objectid', 'globalid', 'survey_date', 'observer', 'reach', 'obs_type', 'species', 'count', 'redd_count', 'water_clarity', 'flow', 'weather', 'notes', 'CreationDate', 'Creator', 'EditDate', 'Editor']


In [13]:
lyr = svc[0].layers[0]
ftype = {"decimal": "esriFieldTypeDouble", "text": "esriFieldTypeString"}
existing = {f["name"] for f in lyr.properties.fields}
new = []
for sf in schema["fields"]:
    if sf.get("system") and sf["name"] not in existing:
        d = {"name": sf["name"], "alias": sf["label"], "type": ftype[sf["type"]], "nullable": True, "editable": True}
        if sf["type"] == "text":
            d["length"] = sf.get("max_length", 255)
        new.append(d)
print(new)
print(lyr.manager.add_to_definition({"fields": new}))

[{'name': 'gps_accuracy_m', 'alias': 'GPS horizontal accuracy (m)', 'type': 'esriFieldTypeDouble', 'nullable': True, 'editable': True}, {'name': 'qa_flag', 'alias': 'QA flag', 'type': 'esriFieldTypeString', 'nullable': True, 'editable': True, 'length': 255}]
{'success': True}


In [15]:
[i for i in gis.content.search('title:"Whatcom Creek Salmon Survey"') if "View Service" in i.typeKeywords][0].homepage

'https://www.arcgis.com/home/item.html?id=75bff57275f64d9fb7887060227ab677'